In [19]:
import re
from rank_bm25 import BM25Okapi

from rageval.schema import RelevantContext, EvalCase, RetrievedChunk
from rageval.metrics.retrieval import (
    relevance_grades,
    hit_at_k,
    ndcg_at_k,
    precision_at_k,
    recall_at_k,
    reciprocal_rank,
)

In [7]:
cookbook = """The Great Pineapple Debate. Few pizza toppings divide people like pineapple. Fans love the sweet and salty contrast; critics call it a crime against pizza.

How Yeast Works. Yeast feeds on the sugars in flour and releases carbon dioxide gas. The gas gets trapped in the dough, which makes it puff up and gives the crust its airy texture.

Letting the Dough Rise. Cover the dough and let it rise in a warm spot for 1 to 2 hours, until it has doubled in size.

A Short History of Pizza. Modern pizza was born in Naples, where street vendors sold flatbreads topped with tomato to working families.

Choosing a Pizza Oven. Home ovens top out around 500°F, while dedicated pizza ovens reach 900°F and cook a pizza in about 90 seconds."""

docs = {"cookbook": cookbook}
print(f"{len(cookbook)} characters")

729 characters


In [9]:
def chunk_paragraphs(doc_id: str, text: str) -> list[RetrievedChunk]:
    """Split a document into one chunk per paragraph, recording each span."""
    return [
        RetrievedChunk(
            doc_id=doc_id,
            chunk_id=f"{doc_id}-{i}",
            span=(m.start(), m.end()),
            text=m.group(),
        )
        # Find every span of characters that is not newline
        for i, m in enumerate(re.finditer(r"[^\n]+", text))
    ]


chunks = chunk_paragraphs("cookbook", cookbook)
for c in chunks:
    print(c.chunk_id, c.span, c.text[:45])

cookbook-0 (0, 155) The Great Pineapple Debate. Few pizza topping
cookbook-1 (157, 337) How Yeast Works. Yeast feeds on the sugars in
cookbook-2 (339, 457) Letting the Dough Rise. Cover the dough and l
cookbook-3 (459, 594) A Short History of Pizza. Modern pizza was bo
cookbook-4 (596, 729) Choosing a Pizza Oven. Home ovens top out aro


In [10]:
all(cookbook[c.span[0] : c.span[1]] == c.text for c in chunks)

True

In [11]:
print(chunks[2].text)

Letting the Dough Rise. Cover the dough and let it rise in a warm spot for 1 to 2 hours, until it has doubled in size.


In [12]:
def span_of(text: str, passage: str) -> tuple[int, int]:
    """Return the character span of a passage inside a document."""
    start = text.index(passage)
    return (start, start + len(passage))

answer = "let it rise in a warm spot for 1 to 2 hours, until it has doubled in size."
background = "The gas gets trapped in the dough, which makes it puff up and gives the crust its airy texture."

case = EvalCase(
    case_id="pizza-01",
    question="How long should a pizza dough rise?",
    reference_answer="1 to 2 hours, until doubled in size.",
    relevant_contexts=[
        RelevantContext(doc_id="cookbook", span=span_of(cookbook, answer), text=answer, relevance=3),
        RelevantContext(doc_id="cookbook", span=span_of(cookbook, background), text=background, relevance=1),
    ],
)

for label in case.relevant_contexts:
    print(label.relevance, label.span, label.text[:45])

3 (383, 457) let it rise in a warm spot for 1 to 2 hours, 
1 (242, 337) The gas gets trapped in the dough, which make


In [16]:
def tokenize(text: str) -> list[str]:
    """Lowercase text and split it into words."""
    return re.findall(r"\w+", text.lower())


bm25 = BM25Okapi([tokenize(c.text) for c in chunks])
scores = bm25.get_scores(tokenize(case.question))

ranked = sorted(zip(chunks, scores), key=lambda pair: pair[1], reverse=True)
retrieved = [c.model_copy(update={"score": float(s)}) for c, s in ranked[:5]]

for rank, c in enumerate(retrieved, start=1):
    print(rank, round(c.score, 2), c.chunk_id, c.text[:45])

1 2.31 cookbook-2 Letting the Dough Rise. Cover the dough and l
2 1.28 cookbook-1 How Yeast Works. Yeast feeds on the sugars in
3 0.75 cookbook-4 Choosing a Pizza Oven. Home ovens top out aro
4 0.63 cookbook-3 A Short History of Pizza. Modern pizza was bo
5 0.6 cookbook-0 The Great Pineapple Debate. Few pizza topping


In [18]:
grades = relevance_grades(retrieved, case.relevant_contexts)

for rank, (c, g) in enumerate(zip(retrieved, grades), start=1):
    print(rank, g, c.chunk_id)
print(grades)

1 3 cookbook-2
2 1 cookbook-1
3 0 cookbook-4
4 0 cookbook-3
5 0 cookbook-0
[3, 1, 0, 0, 0]


In [20]:
def score(retrieved: list[RetrievedChunk], labels: list[RelevantContext], k: int = 5) -> dict:
    """Return all retrieved metrics for one ranked list."""
    grades = relevance_grades(retrieved, labels)
    return {
        f"hit@{k}": hit_at_k(grades, k),
        f"precision@{k}": precision_at_k(grades, k),
        f"recall@{k}": recall_at_k(retrieved, labels, k),
        "reciprocal_rank": reciprocal_rank(grades),
        f"ndcg@{k}": ndcg_at_k(grades, labels, k),
    }

score(retrieved, case.relevant_contexts)

{'hit@5': 1.0,
 'precision@5': 0.4,
 'recall@5': 1.0,
 'reciprocal_rank': 1.0,
 'ndcg@5': 1.0}

In [21]:
bad = list(reversed(retrieved))

for rank, c in enumerate(bad, start=1):
    print(rank, c.chunk_id)

print("k=5", score(bad, case.relevant_contexts))
print("k=2", score(bad, case.relevant_contexts, k=2))

1 cookbook-0
2 cookbook-3
3 cookbook-4
4 cookbook-1
5 cookbook-2
k=5 {'hit@5': 1.0, 'precision@5': 0.4, 'recall@5': 1.0, 'reciprocal_rank': 0.25, 'ndcg@5': 0.4382444959756785}
k=2 {'hit@2': 0.0, 'precision@2': 0.0, 'recall@2': 0.0, 'reciprocal_rank': 0.25, 'ndcg@2': 0.0}
